# Reaction SMILES to an FF path and guess

Install `uv sync --extra ff --extra viz`, then run this notebook from `examples/`. Configure one guesser and supply a reaction SMILES. Each example first computes a path and candidate, then plots and renders them using Plotprofile and xyzrender. These are FF guesses, not verified transition states.

The metal example supplies CXSMILES to specify matching ligand sites. A workflow can also pass its selected rxembed endpoints directly.


In [ ]:
from pathlib import Path

import xyzrender
from IPython.display import display

import rxts

rxts.set_verbose()  # lambda table and FF summary
images = Path("images")
images.mkdir(exist_ok=True)

## Halide substitution

The call maps, embeds and interpolates the reaction. The default sampling budget is:

| Setting | Meaning |
|---|---|
| `n_lambda=21` | Scan frames |
| `n_conformers=4` | Initial spectator-torsion trials, including the starting geometry |
| `n_embed=4` | Requested rxembed proposals per window near the peak |
| `md_steps=0` | No MD; positive values set steps per MD proposal |

Counts are requests, not a guarantee of distinct retained conformers. Set `n_conformers=1, n_embed=0` for an unsampled scan. Set `embed_score="gxtb"` or supply an ASE calculator to rank the sampled peak conformers physically. Finished frames can be scored separately below.


In [ ]:
reaction = "CBr.[Cl-]>>CCl.[Br-]"
guesser = rxts.TransitionStateGuesser(engine="gfnff", multiplicity=1, n_cores=1)
path = guesser.find_transition_state(reaction)
guess = path.guess("ff")
path.dump(images / "path.xyz")
guess.dump(images / "guess.xyz");

In [ ]:
pairs = [(i + 1, j + 1) for i, j in path.forming_bonds + path.breaking_bonds]
breaking = [f"{i + 1}-{j + 1}" for i, j in path.breaking_bonds]
for label, mol in (("reactant", path.reactant), ("product", path.product)):
    display(xyzrender.render(mol, hy=True, canvas_size=420, output=images / f"{label}.svg"))
display(
    xyzrender.render(guess.mol, ts_bonds=pairs, unbond=breaking, hy=True, canvas_size=420, output=images / "guess.svg")
)
display(xyzrender.render_gif(images / "path.xyz", gif_trj=True, trj_bonds=True, hy=True, output=images / "path.gif"))

Optional g-xTB scoring evaluates the existing halide-substitution frames without moving them. Run the next cell to add a second energy series and select its maximum as a candidate; this is not a verified transition state.

In [ ]:
# Use xtb on PATH / XTB_EXE, or replace None with "/path/to/xtb".
rxts.set_xtb(None)
scores = path.score(method="gxtb")
if scores.failures:
    raise RuntimeError(scores.failures)
physical_guess = path.guess("qm")
physical_guess.dump(images / "gxtb_guess.xyz");

In [ ]:
# Toy Lennard-Jones energies demonstrate calculator injection; they are not chemical energies.
from ase.calculators.lj import LennardJones

calculator = LennardJones()
toy_guesser = rxts.TransitionStateGuesser(engine="gfnff", multiplicity=1, n_cores=1)
toy_guesser.embed_score = calculator
toy_path = toy_guesser.find_transition_state("CBr.[Cl-]>>CCl.[Br-]")
toy_scores = toy_path.score(method=calculator)
if toy_scores.failures:
    raise RuntimeError(toy_scores.failures)
selection = toy_path.conformer_selections[0]
selection["selected_index"], selection["scores"].energies_ev

In [ ]:
from plotprofile import ReactionProfilePlotter

plotter = ReactionProfilePlotter(units="kcal", energy="E", labels=False, curviness=0, y2={"label": "bond length (Å)"})
plotter.plot(
    rxts.profile_data(path),
    secondary=rxts.bond_data(path),
    x=[frame.lam for frame in path.frames],
    filename=str(images / "profile"),
    file_format="png",
)

### MD sampling with MMFF

Run the same substitution on `engine="mmff"`, using Langevin proposals instead of peak re-embedding. Both endpoint graphs must have MMFF94 parameters; an unsupported request raises an error. The default `n_conformers=4` requests three new MD proposals at every λ frame, each lasting `md_steps=50` steps. This setting also controls the initial spectator-torsion search. `n_embed=0` disables the later rxembed search; a positive value would enable it alongside MD. Proposals are relaxed on the frozen reaction fields before selection.

The main example uses GFN-FF. `engine="auto"` explicitly permits MMFF where both endpoints are supported, otherwise GFN-FF; inspect `path.engine` for the resolved choice. `engine="uff"` is not implemented for interpolation. UFF in rxembed's geometry construction is separate from the reaction FF.


In [ ]:
md_guesser = rxts.TransitionStateGuesser(multiplicity=1, n_cores=1)
md_guesser.engine = "mmff"
md_guesser.n_conformers = 4
md_guesser.n_embed = 0
md_guesser.md_steps = 50
md_path = md_guesser.find_transition_state(reaction)
md_guess = md_path.guess("ff")
md_path.dump(images / "md_path.xyz")
md_guess.dump(images / "md_guess.xyz");

In [ ]:
pairs = [(i + 1, j + 1) for i, j in md_path.forming_bonds + md_path.breaking_bonds]
breaking = [f"{i + 1}-{j + 1}" for i, j in md_path.breaking_bonds]
display(xyzrender.render(md_guess.mol, ts_bonds=pairs, unbond=breaking, hy=True, output=images / "md_guess.svg"))
display(
    xyzrender.render_gif(images / "md_path.xyz", gif_trj=True, trj_bonds=True, hy=True, output=images / "md_path.gif")
)
plotter.plot(
    rxts.profile_data(md_path),
    secondary=rxts.bond_data(md_path),
    x=[frame.lam for frame in md_path.frames],
    filename=str(images / "md_profile"),
    file_format="png",
)

## Flexible substitution

The same SMILES API works with a larger sampling budget. Each accepted proposal is relaxed on the scan's frozen FFs.


In [ ]:
flexible_guesser = rxts.TransitionStateGuesser(
    engine="gfnff",
    multiplicity=1,
    n_cores=1,
    n_conformers=12,
    n_embed=8,
)
complex_path = flexible_guesser.find_transition_state("OCCCCBr.[Cl-]>>OCCCCCl.[Br-]")
complex_guess = complex_path.guess("ff")
complex_path.dump(images / "complex_path.xyz")
complex_guess.dump(images / "complex_guess.xyz");

In [ ]:
pairs = [(i + 1, j + 1) for i, j in complex_path.forming_bonds + complex_path.breaking_bonds]
breaking = [f"{i + 1}-{j + 1}" for i, j in complex_path.breaking_bonds]
for side, mol in (("reactant", complex_path.reactant), ("product", complex_path.product)):
    display(xyzrender.render(mol, hy=True, canvas_size=420, output=images / f"complex_{side}.svg"))
display(
    xyzrender.render(
        complex_guess.mol,
        ts_bonds=pairs,
        unbond=breaking,
        hy=True,
        canvas_size=420,
        output=images / "complex_guess.svg",
    )
)
display(
    xyzrender.render_gif(
        images / "complex_path.xyz", gif_trj=True, trj_bonds=True, hy=True, output=images / "complex_path.gif"
    )
)

plotter.plot(
    rxts.profile_data(complex_path),
    secondary=rxts.bond_data(complex_path),
    x=[frame.lam for frame in complex_path.frames],
    filename=str(images / "complex_profile"),
    file_format="png",
)

### Optional: supply a selected hydrogen-bond geometry

Geometry control belongs to rxembed. To start this same reaction with an alcohol H···Cl contact, use the following recipe instead of the SMILES call. The contact fixes the starting geometry only; it is not held during the scan.

```python
from rdkit import Chem
import rxembed as rx

mol = Chem.AddHs(rx.parse_smiles("OCCCCBr.[Cl-]"))
h = rx.match(mol, "[O]-[H]")[1]
cl = rx.match(mol, "[Cl-]")[0]
reactant = rx.embed(mol, charge=-1, n=1, threads=1, fix={(h, cl): 2.2})[0]
contact_path = flexible_guesser.find_transition_state(reactant, "OCCCCCl.[Br-]")
contact_guess = contact_path.guess("ff")
```

`path.to_atoms()` and `guess.to_atoms()` export the existing frames or selected candidate as ASE objects, preserving atom order, atom maps, charge, multiplicity and isotope masses for a workflow.


## Hydrolysis, directly and through water

The same formamide hydrolysis can be constructed directly (four centres) or through an added water relay (six centres). `add_relay="O"` adds that water to both sides. The direct reaction goes straight to the guesser. `mechanisms(...)` is used below to construct the added relay route.

In [ ]:
reaction = "NC=O.O>>N.O=CO"
direct_path = guesser.find_transition_state(reaction)
direct_guess = direct_path.guess("ff")
direct_path.dump(images / "direct_path.xyz")
direct_guess.dump(images / "direct_guess.xyz");

In [ ]:
diagram = rxts.draw_mechanisms(reaction)
diagram.save(images / "direct_changes.png")
display(diagram)
plotter.plot(
    rxts.profile_data(direct_path),
    secondary=rxts.bond_data(direct_path),
    x=[frame.lam for frame in direct_path.frames],
    filename=str(images / "direct_profile"),
    file_format="png",
)

In [ ]:
for side, mol in (("reactant", direct_path.reactant), ("product", direct_path.product)):
    display(xyzrender.render(mol, hy=True, output=images / f"direct_{side}.svg"))
pairs = [(i + 1, j + 1) for i, j in direct_path.forming_bonds + direct_path.breaking_bonds]
breaking = [f"{i + 1}-{j + 1}" for i, j in direct_path.breaking_bonds]
display(
    xyzrender.render(direct_guess.mol, ts_bonds=pairs, unbond=breaking, hy=True, output=images / "direct_guess.svg")
)
display(
    xyzrender.render_gif(
        images / "direct_path.xyz", gif_trj=True, trj_bonds=True, hy=True, output=images / "direct_path.gif"
    )
)

### Add a water relay

Only the `add_relay` request changes. The additional water carries the proton between the reacting water and the leaving amine.

In [ ]:
relay_route = rxts.mechanisms(reaction, add_relay="O")[0]
relay_path = guesser.find_transition_state(relay_route["smiles"])
relay_guess = relay_path.guess("ff")
relay_path.dump(images / "relay_path.xyz")
relay_guess.dump(images / "relay_guess.xyz");

In [ ]:
diagram = rxts.draw_mechanisms(relay_route["smiles"])
diagram.save(images / "relay_changes.png")
display(diagram)
plotter.plot(
    rxts.profile_data(relay_path),
    secondary=rxts.bond_data(relay_path),
    x=[frame.lam for frame in relay_path.frames],
    filename=str(images / "relay_profile"),
    file_format="png",
)

In [ ]:
for side, mol in (("reactant", relay_path.reactant), ("product", relay_path.product)):
    display(xyzrender.render(mol, hy=True, output=images / f"relay_{side}.svg"))
pairs = [(i + 1, j + 1) for i, j in relay_path.forming_bonds + relay_path.breaking_bonds]
breaking = [f"{i + 1}-{j + 1}" for i, j in relay_path.breaking_bonds]
display(xyzrender.render(relay_guess.mol, ts_bonds=pairs, unbond=breaking, hy=True, output=images / "relay_guess.svg"))
display(
    xyzrender.render_gif(
        images / "relay_path.xyz", gif_trj=True, trj_bonds=True, hy=True, output=images / "relay_path.gif"
    )
)

### Use water already included in the reaction

`relay="O"` selects an unchanged water already present on both sides; it adds no atoms. The reacting water is excluded. For two existing relay waters, use `relay="O.O"`. To choose particular mapped protic atoms, use `relay_map=99` or `relay_map=(99, 100)`.

In [ ]:
full_reaction = "NC=O.O.O>>N.O=CO.O"
existing_route = rxts.mechanisms(full_reaction, relay="O")[0]
display(rxts.draw_mechanisms(existing_route["smiles"]))
# Run guesser.find_transition_state(existing_route["smiles"]), as above.

### Two-water proton chain

A dot-separated relay request uses both waters in the chain (eight centres). Adding and selecting remain separate operations:

In [ ]:
two_added = rxts.mechanisms(reaction, add_relay="O.O")[0]
two_present = rxts.mechanisms("NC=O.O.O.O>>N.O=CO.O.O", relay="O.O")[0]
display(rxts.draw_mechanisms(two_added["smiles"]))
display(rxts.draw_mechanisms(two_present["smiles"]))

## Beta-hydride elimination in a Pd Heck cycle

The same call accepts CXSMILES. These strings specify a square-planar state with corresponding ligand sites on both sides:

| Site | Reactant | Product |
|---|---|---|
| `s0` | phosphine 5 | phosphine 5 |
| `s1` | alkyl 1 | η² alkene 1–2, re face |
| `s2` | vacant | hydride 3 |
| `s3` | phosphine 6 | phosphine 6 |

The mapped beta hydrogen 3 is explicit in both strings. The `atomProp` block stores the rxembed shape and site labels; its indices address each string's atoms, while atom maps identify corresponding atoms across the reaction. Ordinary SMILES does not specify this seating. Supply chosen CXSMILES or selected rxembed isomers/conformers rather than choosing an isomer implicitly.

These are constructed endpoints illustrating the cationic step, not the paper's optimized geometries or verified saddles. [van Hoorn et al. (2026)](https://doi.org/10.1002/jcc.70454).


In [ ]:
reactant = (
    "C[P:5](C)(C)->[Pd+2:4](<-[CH2-:1][CH:2]([H:3])c1ccccc1)<-[P:6](C)(C)C "
    "|atomProp:1.atomNote.s0:4.atomNote.SPL:5.atomNote.s1:14.atomNote.s3|"
)
product = (
    "C[P:5](C)(C)->[Pd+2:4]1(<-[CH2:1]=[CH:2]->1c1ccccc1)(<-[H-:3])<-[P:6](C)(C)C "
    "|atomProp:1.atomNote.s0:4.atomNote.SPL:5.atomNote.s1+:6.atomNote.s1+:13.atomNote.s2:14.atomNote.s3|"
)
beta_path = guesser.find_transition_state(reactant, product)
beta_guess = beta_path.guess("ff")
beta_path.dump(images / "beta_path.xyz")
beta_guess.dump(images / "beta_guess.xyz");

In [ ]:
pairs = [(i + 1, j + 1) for i, j in beta_path.forming_bonds + beta_path.breaking_bonds]
breaking = [f"{i + 1}-{j + 1}" for i, j in beta_path.breaking_bonds]
for side, mol in (("reactant", beta_path.reactant), ("product", beta_path.product)):
    display(xyzrender.render(mol, hy=True, canvas_size=420, output=images / f"beta_{side}.svg"))
display(
    xyzrender.render(
        beta_guess.mol, ts_bonds=pairs, unbond=breaking, hy=True, canvas_size=420, output=images / "beta_guess.svg"
    )
)
display(
    xyzrender.render_gif(
        images / "beta_path.xyz", gif_trj=True, trj_bonds=True, hy=True, output=images / "beta_path.gif"
    )
)

plotter.plot(
    rxts.profile_data(beta_path),
    secondary=rxts.bond_data(beta_path),
    x=[frame.lam for frame in beta_path.frames],
    filename=str(images / "beta_profile"),
    file_format="png",
)